In [18]:
import geopandas as gpd
from rasterstats import zonal_stats
import pandas as pd

In [19]:
# 1. Cargar las subcuencas
cuencas = gpd.read_file('../qgis_status_outlook/shapefiles/Cuencas_n2.shp')
# from cuencas, extract cod_n2, nombrec2, area and create a dataframe
cuencas_df = cuencas[['cod_n2', 'nombrec2', 'area']].copy()
cuencas_df.rename(columns={'cod_n2': 'cod_n2', 'nombrec2': 'nombrec2', 'area': 'area'}, inplace=True)
cuencas_df.head()

,cod_n2,nombrec2,area
0,10,RÍO CUAREIM,8222
1,11,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,2583
2,12,RÍO ARAPEY CHICO,2152
3,13,RÍO ARAPEY GRANDE,9698
4,14,RÍO URUGUAY entre Río Arapey Grande y Río Daymán,1631


In [20]:
# 2. Definir los archivos raster de SPI
# De la carpeta ../qgis_status_outlook/IPE/ extraer los archivos raster de SPI para los periodos 1, 3, 6 y 12 meses. 
# Los nombres de archivo contienen el periodo de tiempo, por ejemplo: YYYY_MM_IPE_01meses, YYYY_MM_IPE_03meses, YYYY_MM_IPE_06meses, YYYY_MM_IPE_12meses. Donde YYYY es el año y MM es el mes.
ano = '2026'
mes = '02'

raster_files = {
    '1_mes': f'../qgis_status_outlook/IPE/{ano}_{mes}_IPE_01meses.tif',
    '3_meses': f'../qgis_status_outlook/IPE/{ano}_{mes}_IPE_03meses.tif',
    '6_meses': f'../qgis_status_outlook/IPE/{ano}_{mes}_IPE_06meses.tif',
    '12_meses': f'../qgis_status_outlook/IPE/{ano}_{mes}_IPE_12meses.tif'
}

In [21]:
import os

# 3. Iterar sobre cada raster y calcular el promedio
for col_name, raster_path in raster_files.items():
    print(f"Procesando: {col_name} - {raster_path}")
    
    # Verificar que el archivo existe
    if not os.path.exists(raster_path):
        print(f"Archivo no encontrado: {raster_path}")
        cuencas[col_name] = None
        continue
    
    # zonal_stats devuelve una lista de diccionarios
    # Asegurar que ambos tienen el mismo CRS y usar all_touched=True para capturar valores
    # Usar band=2 para especificar la banda 2
    stats = zonal_stats(
        cuencas.to_crs(epsg=32721),  # Asegurar CRS común
        raster_path, 
        stats="mean",
        all_touched=True,
        nodata=-999,
        band=1  # Usar la banda 1 del raster
    )
    
    # Extraemos el valor 'mean' y lo asignamos a la columna cod_n2 de el dataframe cuencas_df
    cuencas_df[col_name] = [x['mean'] if x['mean'] is not None else float('nan') for x in stats]
    
    # Mostrar resumen
    valid_values = sum(1 for x in stats if x['mean'] is not None)
    print(f"Valores válidos: {valid_values}/{len(stats)}")

Procesando: 1_mes - ../qgis_status_outlook/IPE/2026_02_IPE_01meses.tif
Valores válidos: 47/47
Procesando: 3_meses - ../qgis_status_outlook/IPE/2026_02_IPE_03meses.tif
Valores válidos: 47/47
Procesando: 6_meses - ../qgis_status_outlook/IPE/2026_02_IPE_06meses.tif
Valores válidos: 47/47
Procesando: 12_meses - ../qgis_status_outlook/IPE/2026_02_IPE_12meses.tif
Valores válidos: 47/47


In [22]:
# import 01_status_month.csv, 03_status_month.csv
status_1_mes = pd.read_csv('../qgis_status_outlook/csvtables/01_status_month.csv')
status_3_meses = pd.read_csv('../qgis_status_outlook/csvtables/03_status_month.csv')


In [23]:
# 4. asignar el status_1_mes y status_3_meses a cuencas_df usando cod_n2 como clave
cuencas_df = cuencas_df.merge(
    status_1_mes.rename(columns={'stationID': 'cod_n2', 'category': 'status_1_mes'}),
    on='cod_n2', how='left'
)
cuencas_df = cuencas_df.merge(
    status_3_meses.rename(columns={'stationID': 'cod_n2', 'category': 'status_3_meses'}),
    on='cod_n2', how='left'
)
cuencas_df.head()

,cod_n2,nombrec2,area,1_mes,3_meses,6_meses,12_meses,status_1_mes,status_3_meses
0,10,RÍO CUAREIM,8222,-0.697120,-0.062037,-0.068046,0.237027,3,3
1,11,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,2583,-0.724263,-0.111608,-0.137099,0.008922,3,3
2,12,RÍO ARAPEY CHICO,2152,-0.724263,-0.138935,-0.150500,0.024353,3,3
3,13,RÍO ARAPEY GRANDE,9698,-0.724263,-0.282703,-0.305399,-0.195424,3,3
4,14,RÍO URUGUAY entre Río Arapey Grande y Río Daymán,1631,-0.655983,-0.154178,-0.262393,-0.302497,3,3


In [26]:
# 5. Clasificar IPE en categorías (0-4) según umbrales
import numpy as np

def clasificar_ipe(valor):
    if np.isnan(valor):
        return np.nan
    if valor >= -0.5:
        return 0
    elif valor >= -1.0:
        return 1
    elif valor >= -1.5:
        return 2
    elif valor >= -2.0:
        return 3
    else:
        return 4

cuencas_df['IPE_1'] = cuencas_df['1_mes'].apply(clasificar_ipe)
cuencas_df['IPE_3'] = cuencas_df['3_meses'].apply(clasificar_ipe)
cuencas_df['IPE_6'] = cuencas_df['6_meses'].apply(clasificar_ipe)
cuencas_df['IPE_12'] = cuencas_df['12_meses'].apply(clasificar_ipe)


# drop columns 1_mes, 3_meses, 6_meses, 12_meses
cuencas_df.drop(columns=['1_mes', '3_meses', '6_meses', '12_meses'], inplace=True)

cuencas_df.head()

,cod_n2,nombrec2,area,status_1_mes,status_3_meses,IPE_1,IPE_3,IPE_6,IPE_12
0,10,RÍO CUAREIM,8222,3,3,1,0,0,0
1,11,RÍO URUGUAY entre Río Cuareim y Río Arapey Grande,2583,3,3,1,0,0,0
2,12,RÍO ARAPEY CHICO,2152,3,3,1,0,0,0
3,13,RÍO ARAPEY GRANDE,9698,3,3,1,0,0,0
4,14,RÍO URUGUAY entre Río Arapey Grande y Río Daymán,1631,3,3,1,0,0,0
